# 02. Ekstrakcja cech i walidacja jakości danych (Warstwa Silver)
### Instalacja wymaganych bibliotek
Instalujemy pakiety do obsługi obrazów i głębokiego uczenia (`pillow`, `torch`, `torchvision`, `numpy`, `pyyaml`), a następnie restartujemy środowisko Pythona.

In [0]:
%pip install -U pillow pyyaml numpy torch torchvision
dbutils.library.restartPython()

### Wczytanie konfiguracji i pobranie metadanych bieżącej paczki z warstwy Bronze
Odczytujemy numer aktualnie przetwarzanej paczki z tabeli `pipeline_state` i pobieramy odpowiadające jej rekordy (ścieżki do plików, etykiety klas oraz jasność) z tabeli `bronze_batch_stats`.

In [0]:
import yaml
import os

config_path = "config.yaml" if os.path.exists("config.yaml") else "../config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

CATALOG = config["storage"]["catalog_name"]
SCHEMA = config["storage"]["schema_name"]
HF_DATASET = config["dataset"]["hf_name"]
REPLAY_MAX_SIZE = config["pipeline"]["replay_buffer_size"]

# Sprawdzamy w pipeline_state, który numer paczki jest teraz przetwarzany
state_row = spark.sql(f"""
    SELECT last_batch_id 
    FROM {CATALOG}.{SCHEMA}.pipeline_state 
    WHERE dataset_name = '{HF_DATASET}'
    ORDER BY last_batch_id DESC LIMIT 1
""").collect()[0]

current_batch_id = state_row["last_batch_id"] + 1

# Pobieramy rekordy z tabeli Bronze dla tej paczki
bronze_rows = spark.sql(f"""
    SELECT batch_id, global_idx, file_path, label_id, class_name, brightness
    FROM {CATALOG}.{SCHEMA}.bronze_batch_stats
    WHERE batch_id = {current_batch_id}
""").collect()

print(f"Przetwarzanie warstwy Silver dla Paczki #{current_batch_id} (znaleziono {len(bronze_rows)} zdjęć w Bronze).")

### Kontrola jakości obrazów (Data Quality) i ekstrakcja wektorów cech (`MobileNetV3-Small`)
Sprawdzamy każdy obraz pod kątem jakości (czy plik istnieje, nie jest uszkodzony oraz czy jego średnia jasność mieści się w przedziale `5.0 - 250.0`, odrzucając całkowicie czarne lub prześwietlone zdjęcia). Poprawne obrazy przepuszczamy w paczkach przez zamrożoną sieć `MobileNetV3-Small`, uzyskując 576-wymiarowe wektory cech (embeddingi).

In [0]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from datetime import datetime

# 1. Przygotowanie zamrożonego ekstraktora MobileNetV3-Small
weights = models.MobileNet_V3_Small_Weights.DEFAULT
backbone = models.mobilenet_v3_small(weights=weights)
backbone.classifier = nn.Identity()
for param in backbone.parameters():
    param.requires_grad = False
backbone.eval()

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

silver_records = []
tensors_batch = []
meta_batch = []
rejected_count = 0

with torch.no_grad():
    for idx, row in enumerate(bronze_rows):
        file_path = row["file_path"]
        brightness = row["brightness"]
        
        # KONTROLA JAKOŚCI (Silver Data Quality):
        # 1. Czy plik istnieje? 2. Czy nie jest całkowicie czarny (<5) lub biały (>250)?
        if not os.path.exists(file_path) or brightness < 5.0 or brightness > 250.0:
            rejected_count += 1
            continue
            
        try:
            img = Image.open(file_path).convert("RGB")
            tensors_batch.append(preprocess(img))
            meta_batch.append(row)
        except Exception as e:
            print(f"Odrzucono uszkodzony plik {file_path}: {e}")
            rejected_count += 1
            continue
            
        # Przepuszczamy przez sieć w paczkach po 64 zdjęcia
        if len(tensors_batch) == 64 or (idx + 1) == len(bronze_rows):
            if len(tensors_batch) > 0:
                batch_tensor = torch.stack(tensors_batch)
                embeddings = backbone(batch_tensor).numpy()
                
                for meta, emb in zip(meta_batch, embeddings):
                    silver_records.append({
                        "batch_id": int(meta["batch_id"]),
                        "global_idx": int(meta["global_idx"]),
                        "label_id": int(meta["label_id"]),
                        "class_name": meta["class_name"],
                        "features": [float(x) for x in emb],
                        "processed_at": datetime.utcnow()
                    })
                tensors_batch, meta_batch = [], []

print(f"Walidacja zakończona! Poprawnych zdjęć: {len(silver_records)}, Odrzuconych: {rejected_count}")

### Zapis wektorów bieżącej paczki (`silver_current_batch`) i aktualizacja bufora pamięci (`silver_replay_buffer`)
Zapisujemy wyekstrahowane wektory cech bieżącej paczki w tabeli `silver_current_batch`. Dodatkowo losujemy próbkę wektorów i aktualizujemy tabelę `silver_replay_buffer` (Experience Replay), która zapobiega zjawisku katastrofalnego zapominania (*catastrophic forgetting*) podczas douczania modeli na kolejnych paczkach.

In [0]:
import random

# 1. Zapisujemy wektory aktualnej paczki (overwrite - bo tabela trzyma zawsze tylko bieżącą paczkę do nauki)
silver_df = spark.createDataFrame(silver_records)
silver_df.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_current_batch")
print(f"Zapisano {len(silver_records)} wektorów w tabeli {CATALOG}.{SCHEMA}.silver_current_batch.")

# 2. Aktualizacja Replay Buffera (losujemy np. 30 przykładów z obecnej paczki)
sample_size = min(30, len(silver_records))
replay_candidates = random.sample(silver_records, sample_size)

# Jeśli tabela silver_replay_buffer już istnieje, łączymy stare próbki z nowymi i zostawiamy max REPLAY_MAX_SIZE (100)
if spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.silver_replay_buffer"):
    existing_replay = [row.asDict() for row in spark.table(f"{CATALOG}.{SCHEMA}.silver_replay_buffer").collect()]
    combined_replay = existing_replay + replay_candidates
    if len(combined_replay) > REPLAY_MAX_SIZE:
        combined_replay = random.sample(combined_replay, REPLAY_MAX_SIZE)
else:
    combined_replay = replay_candidates

replay_df = spark.createDataFrame(combined_replay)
replay_df.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_replay_buffer")
print(f"Zaktualizowano bufor pamięci {CATALOG}.{SCHEMA}.silver_replay_buffer (aktualny rozmiar: {len(combined_replay)} wektorów).")

# Podgląd pierwszych 5 wierszy z wyliczonymi wektorami cech
display(spark.sql(f"SELECT batch_id, global_idx, label_id, class_name, features FROM {CATALOG}.{SCHEMA}.silver_current_batch LIMIT 5"))